# TTM-Small-4M × EmbeddingGemma 2 — Causal6 (6 nhánh)

**Tiền đăng ký đã khóa, Colab Free Tesla T4, FP32.** Dữ liệu H512 phải được niêm phong trên Windows trước. Không sử dụng lại TEST 64 truyện hoặc holdout 320 truyện L0017.

1. Tải ZIP riêng tư từ máy Windows lên `MyDrive/TTM/ttm-small-pilot/TTM_GEMMA2_CAUSAL6_INPUT_PRIVATE_V1_20261109.zip` (không đặt công khai).
2. Chọn GPU Tesla T4 trong Colab Free. Điền SHA256 ZIP do PowerShell in ra và dung lượng Drive còn trống **thực sự kiểm tra từ tài khoản**.
3. Chạy hai ô đầu để xác minh. Ô thứ ba là lần **CLAIM + huấn luyện thật**; chỉ chạy khi ô PRE-CLAIM PASS. Không nhấn chạy lại sau CLAIM hoặc khi gián đoạn; `resume` chỉ được phép kiểm tra từ READY hợp lệ cùng RunID.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, os, shutil, zipfile, subprocess, sys
# Owner fills EXACT bytes SHA256 printed by the Windows stage script.
EXPECTED_PRIVATE_ZIP_SHA256 = ''
OWNER_VERIFIED_DRIVE_FREE_GIB = 0.0  # GiB checked in Google Drive, NOT guessed
OWNER_FREE_COLAB_ONLY = True
MODE = 'execute'  # For later adjudicated continuation change to 'resume' only if READY verified
assert len(EXPECTED_PRIVATE_ZIP_SHA256)==64, 'HOLD_ENTER_WINDOWS_PRIVATE_ZIP_SHA256'
assert OWNER_VERIFIED_DRIVE_FREE_GIB >= 4, 'HOLD_CHECK_ACTUAL_GOOGLE_DRIVE_QUOTA'
assert OWNER_FREE_COLAB_ONLY, 'HOLD_FREE_ONLY'
assert MODE in ('execute','resume')
drive.mount('/content/drive')
source_zip = Path('/content/drive/MyDrive/TTM/ttm-small-pilot/TTM_GEMMA2_CAUSAL6_INPUT_PRIVATE_V1_20261109.zip')
assert source_zip.is_file(), 'HOLD_OWNER_PRIVATE_INPUT_NOT_UPLOADED'
h=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert h(source_zip)==EXPECTED_PRIVATE_ZIP_SHA256.lower(), 'HOLD_PRIVATE_PACKAGE_SHA256'
work = Path('/content/ttm_causal6_source_v1')
assert not work.exists(), 'HOLD_TEMP_SOURCE_EXISTS_NO_OVERWRITE'
work.mkdir(parents=True,exist_ok=False)
with zipfile.ZipFile(source_zip) as z:
    assert z.testzip() is None, 'ZIP_CRC_ERROR'
    for item in z.infolist():
        rel = Path(item.filename)
        assert not rel.is_absolute() and '..' not in rel.parts and rel.parts[0] in ('input','code'), 'ZIP_PATH_REFUSED'
    z.extractall(work)
input_dir=work/'input'
h512=input_dir/'h512'
runner=work/'code'/'ttm_causal6_colab_free_t4_one_shot.py'
reviewer=work/'code'/'ttm_causal6_independent_readback.py'
assert all(p.exists() for p in (runner,reviewer,h512/'H512_PUBLIC_SEAL.json'))
run=Path('/content/drive/MyDrive/TTM/ttm-small-pilot/ttm_small4m_gemma2_causal6_2048x4096_free_t4_20261109_v1')
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('TTM_CAUSAL6_SOURCE_ZIP=VERIFIED')


In [ ]:
args=[sys.executable,'-B',str(runner),'--mode','preflight','--input',str(input_dir),'--h512',str(h512),
      '--drive-output',str(run),'--owner-verified-drive-free-gib',str(OWNER_VERIFIED_DRIVE_FREE_GIB),'--free-only-confirmed']
if MODE=='resume':
    # Refuse an unreviewed resume. Re-run only after evidence shows all chunks through READY; no uncommitted progress.
    raise RuntimeError('HOLD_RESUME_REQUIRES_EXPLICIT_ADJUDICATION_AND_MANUAL_COMMAND')
assert not run.exists(), 'RUN_ALREADY_HAS_CLAIM_OR_EVIDENCE'
subprocess.run(args,check=True)
print('TTM_CAUSAL6_PRECLAIM=PASS_NO_SCIENCE_CONSUMED')


In [ ]:
# THIS CELL CONSUMES THE ONE-SHOT. Never rerun after CLAIM (including after interruption).
assert MODE=='execute' and not run.exists(), 'HOLD_ALREADY_CLAIMED_OR_NOT_INITIAL'
cmd=[sys.executable,'-B',str(runner),'--mode','execute','--input',str(input_dir),'--h512',str(h512),
     '--drive-output',str(run),'--owner-verified-drive-free-gib',str(OWNER_VERIFIED_DRIVE_FREE_GIB),'--free-only-confirmed']
subprocess.run(cmd,check=True)
assert (run/'RESULT.json').is_file(), 'HOLD_SCIENCE_INCOMPLETE_DO_NOT_RERUN'
subprocess.run([sys.executable,'-B',str(reviewer),'--mode','readback','--run',str(run)],check=True)
print('TTM_CAUSAL6_COMPLETED_PENDING_INDEPENDENT_ADJUDICATION')
print((run/'RESULT.json').read_text())
print((run/'INDEPENDENT_READBACK_PUBLIC.json').read_text())
